# 03 · Chain rule by hand and gradient checking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/math/03_chain_rule_gradcheck.ipynb) &nbsp; 📖 Concept deck: [Derivatives and the chain rule](https://nilesh-salpe.github.io/visual-ai/math/04-derivatives-chain-rule/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** forward and backward passes of a small computational graph by hand, the gradient of a linear layer with a squared loss, and a finite-difference gradient checker.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

## Task 1: Backward through a tiny graph

For $f(x, y) = (x \cdot y + \sin x)^2$, compute the value and the gradient by the chain rule, node by node:
$a = x y$, $b = \sin x$, $c = a + b$, $f = c^2$. Backward: $\partial f/\partial c = 2c$, then $\partial f/\partial x = 2c\,(y + \cos x)$ and $\partial f/\partial y = 2c\,x$.
Return `(f, dfdx, dfdy)`.

In [ ]:
def graph_grads(x, y):
    # TODO: forward a, b, c, f; backward dc, then dx (two paths!) and dy
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_graph_grads(f):
    v, dx, dy = f(1.0, 2.0)
    c = 2 + np.sin(1)
    assert np.isclose(v, c ** 2)
    assert np.isclose(dx, 2 * c * (2 + np.cos(1))), "x reaches f through two paths (a and b): add both"
    assert np.isclose(dy, 2 * c * 1.0)

check("graph_grads", _test_graph_grads, graph_grads)

## Task 2: Gradients of a linear layer

For $\hat Y = X W + b$ and $L = \frac{1}{n}\sum (\hat Y - Y)^2$ (mean over all entries), return `(L, dW, db)`.
Shapes: X is (n, d), W is (d, k), b is (k,). Hint: $\partial L / \partial \hat Y = \frac{2}{nk}(\hat Y - Y)$, then $dW = X^\top \, \partial L/\partial \hat Y$ and $db$ sums over rows.

In [ ]:
def linear_mse_grads(X, W, b, Y):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_linear_mse_grads(f):
    g = np.random.default_rng(0)
    X, W, b, Y = g.normal(size=(5, 3)), g.normal(size=(3, 2)), g.normal(size=2), g.normal(size=(5, 2))
    L, dW, db = f(X, W, b, Y)
    assert np.isclose(L, np.mean((X @ W + b - Y) ** 2))
    assert dW.shape == W.shape and db.shape == b.shape, "a gradient has the shape of its parameter"
    eps = 1e-6; Wp = W.copy(); Wp[1, 0] += eps
    num = (np.mean((X @ Wp + b - Y) ** 2) - L) / eps
    assert abs(num - dW[1, 0]) < 1e-4, "dW disagrees with a finite difference"

check("linear_mse_grads", _test_linear_mse_grads, linear_mse_grads)

## Task 3: A finite-difference gradient checker

Write `grad_check(f, x, eps)` that returns the numerical gradient of a scalar function `f` at array `x` with **central differences**:
$\dfrac{f(x + \epsilon e_i) - f(x - \epsilon e_i)}{2\epsilon}$ for each coordinate i. Central differences have error $O(\epsilon^2)$, one-sided $O(\epsilon)$.

In [ ]:
def grad_check(f, x, eps=1e-5):
    # TODO: loop over coordinates
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_grad_check(f):
    x = np.array([0.5, -1.0, 2.0])
    g = f(lambda v: np.sum(v ** 3), x)
    assert np.allclose(g, 3 * x ** 2, atol=1e-6)
    g2 = f(lambda v: np.sin(v[0]) * v[1], np.array([1.0, 2.0]))
    assert np.allclose(g2, [2 * np.cos(1), np.sin(1)], atol=1e-6)

check("grad_check", _test_grad_check, grad_check)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Backward through a tiny graph
def graph_grads_solution(x, y):
    a, b = x * y, np.sin(x); c = a + b; f = c ** 2
    dc = 2 * c
    dx = dc * y + dc * np.cos(x)
    dy = dc * x
    return f, dx, dy

check("graph_grads (reference)", _test_graph_grads, graph_grads_solution)

In [ ]:
# @title Solution for Task 2: Gradients of a linear layer
def linear_mse_grads_solution(X, W, b, Y):
    Yh = X @ W + b
    L = np.mean((Yh - Y) ** 2)
    dY = 2 * (Yh - Y) / Yh.size
    return L, X.T @ dY, dY.sum(axis=0)

check("linear_mse_grads (reference)", _test_linear_mse_grads, linear_mse_grads_solution)

In [ ]:
# @title Solution for Task 3: A finite-difference gradient checker
def grad_check_solution(f, x, eps=1e-5):
    g = np.zeros_like(x, dtype=float)
    for i in range(x.size):
        e = np.zeros_like(x, dtype=float); e.flat[i] = eps
        g.flat[i] = (f(x + e) - f(x - e)) / (2 * eps)
    return g

check("grad_check (reference)", _test_grad_check, grad_check_solution)

---
## Experiments
**Choosing ε.** Too large and the approximation error dominates; too small and floating-point cancellation does. Central differences are best around ε ≈ 10⁻⁵ in float64.

In [ ]:
f, x, true = np.sin, 1.0, np.cos(1.0)
for eps in [1e-1, 1e-3, 1e-5, 1e-8, 1e-11]:
    one = (f(x + eps) - f(x)) / eps; two = (f(x + eps) - f(x - eps)) / (2 * eps)
    print(f"eps={eps:.0e}: one-sided error {abs(one - true):.1e}   central error {abs(two - true):.1e}")

---
## Takeaways
* The chain rule multiplies local derivatives along each path and adds over paths.
* A gradient always has the shape of the thing it differentiates; dW = Xᵀ·dŶ for a linear layer.
* Check hand-written gradients against central finite differences before trusting them.